[![Buka di Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sihitam/aai-tugas1-4-readmisi-diabetes130/blob/main/notebooks/02_tugas2_praproses_diabetes130.ipynb)

# Praproses: Diabetes 130-US Hospitals (Tugas 2 AAI)

**Kasus:** prediksi risiko pasien diabetes dirawat ulang dalam 30 hari setelah pulang (readmisi 30 hari).
**Data:** UCI *Diabetes 130-US Hospitals for Years 1999-2008* (101.766 kunjungan, 50 kolom), CC BY 4.0.
Sitasi: Clore, J., Cios, K., DeShazo, J., & Strack, B. (2014). UCI Machine Learning Repository. https://doi.org/10.24432/C5230J

Pilihan desain yang bisa diubah ditandai **KEPUTUSAN** dan dikumpulkan di bagian Parameter. Tiap langkah punya bagian "Kenapa" yang menjelaskan alasan pilihannya.

## Alur praproses

| Langkah | Isi |
|---|---|
| 1 | Muat data dan profil singkat |
| 2 | Missing value (`?`) |
| 3 | Bersihkan baris (pasien meninggal/hospice, gender tidak valid, pasien berulang) |
| 4 | Buang kolom (ID, konstan, hampir konstan) |
| 5 | Transformasi (usia, diagnosis, kode admisi dan pulang, label) |
| 6 | Bagi data latih dan uji |
| 7 | Outlier, encoding, scaling (belajar dari data latih saja) |
| 8 | Kelas tidak seimbang (hanya data latih) |
| 9 | Analisis kepentingan fitur (tidak ada fitur dibuang) |
| 10 | Tabel sebelum dan sesudah, pemeriksaan akhir |
| 11 | Simpan hasil |

## Parameter

In [1]:
# KEPUTUSAN 1: bentuk label
#   "biner"  -> 1 = dirawat ulang < 30 hari, 0 = lainnya
#   "3kelas" -> 0 = tidak dirawat ulang, 1 = lebih dari 30 hari, 2 = kurang dari 30 hari
LABEL_MODE = "biner"

# KEPUTUSAN 2: kolom medical_specialty dan payer_code (banyak kosong)
#   "kategori" -> isi kosong dengan 'Unknown' (kolom tetap dipakai)
#   "buang"    -> kolom dibuang
STRATEGI_KOSONG_BESAR = "kategori"

# KEPUTUSAN 3: ambang kolom obat 'hampir konstan' (dibuang bila >= ambang bernilai 'No')
AMBANG_OBAT_KONSTAN = 0.99

RANDOM_STATE = 42

## Langkah 1. Muat data dan profil singkat

In [2]:
import glob, os, io, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)

def cari_data():
    # 1) cari di folder kerja (rekursif) dan di folder data repo
    for pola in ["./**/diabetic_data.csv", "/content/**/diabetic_data.csv"]:
        f = glob.glob(pola, recursive=True)
        if f:
            return f[0]
    for p in ["../data/raw/diabetic_data.csv", "data/raw/diabetic_data.csv"]:
        if os.path.exists(p):
            return p
    # 2) unduh dari repo GitHub ini (untuk Google Colab), lalu 3) dari UCI
    os.makedirs("data_uci", exist_ok=True)
    sumber = [("repo", "https://raw.githubusercontent.com/sihitam/aai-tugas1-4-readmisi-diabetes130/main/data/raw/diabetic_data.csv"),
              ("uci", "https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip")]
    for nama, url in sumber:
        try:
            if nama == "repo":
                urllib.request.urlretrieve(url, os.path.join("data_uci", "diabetic_data.csv"))
            else:
                z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url, timeout=60).read()))
                z.extractall("data_uci")
            return os.path.join("data_uci", "diabetic_data.csv")
        except Exception as e:
            print(f"Unduhan dari {nama} gagal: {e}")
    raise SystemExit("Data tidak ditemukan. Taruh diabetic_data.csv di folder kerja atau di data/raw/.")


PATH = cari_data()
print("Memakai:", PATH)

# keep_default_na=False: 'None' pada A1Cresult/max_glu_serum adalah kategori (tes tidak dilakukan), bukan missing
df_raw = pd.read_csv(PATH, keep_default_na=False)
print("Ukuran awal:", df_raw.shape)

tanda = (df_raw == "?").sum()
tanda = tanda[tanda > 0]
print("\nKolom dengan tanda '?':")
print(pd.DataFrame({"jumlah": tanda, "persen": (tanda / len(df_raw) * 100).round(1)}))
print("\nLabel readmitted:", df_raw["readmitted"].value_counts().to_dict())

Memakai: ../data/raw/diabetic_data.csv


Ukuran awal: (101766, 50)

Kolom dengan tanda '?':
                   jumlah  persen
race                 2273     2.2
weight              98569    96.9
payer_code          40256    39.6
medical_specialty   49949    49.1
diag_1                 21     0.0
diag_2                358     0.4
diag_3               1423     1.4

Label readmitted: {'NO': 54864, '>30': 35545, '<30': 11357}


**Kenapa:** sebelum membersihkan, kita perlu tahu apa yang kotor. Di data ini missing value ditulis `?`, bukan kosong, jadi `isna()` biasa tidak melihatnya.
Pembacaan dengan `keep_default_na=False` menjaga kata `None` tetap sebagai teks karena artinya "tes tidak dilakukan", bukan "data hilang".

## Langkah 2. Missing value

**Kenapa:** model tidak bisa memakai `?`. Keputusan tiap kolom bergantung pada seberapa banyak yang kosong dan apa artinya.
- `weight` 96,9% kosong: tidak ada gunanya diimputasi, kolom dibuang.
- `medical_specialty` (49%) dan `payer_code` (40%): terlalu banyak untuk dihapus barisnya; dijadikan kategori "Unknown" atau kolom dibuang (**KEPUTUSAN 2**).
- `race` (2,2%): kategori "Unknown".
- `diag_1` (21 baris): diagnosis utama tidak boleh kosong, barisnya dibuang.
- `diag_2` dan `diag_3`: kosong dianggap pasien tidak punya diagnosis tambahan, diisi "Tidak ada". **Ini asumsi.**

In [3]:
log = []   # catatan jumlah baris dan kolom tiap langkah, untuk tabel akhir
def catat(langkah, d):
    log.append({"langkah": langkah, "baris": len(d), "kolom": d.shape[1]})

df = df_raw.copy()
catat("Data mentah", df)

df = df.replace("?", np.nan)
df = df.drop(columns=["weight"])

df["race"] = df["race"].fillna("Unknown")
for kol in ["medical_specialty", "payer_code"]:
    if STRATEGI_KOSONG_BESAR == "kategori":
        df[kol] = df[kol].fillna("Unknown")
    else:
        df = df.drop(columns=[kol])

df = df.dropna(subset=["diag_1"])
df[["diag_2", "diag_3"]] = df[["diag_2", "diag_3"]].fillna("Tidak ada")

sisa = df.isna().sum()
print("Sisa missing value:", sisa[sisa > 0].to_dict() or "tidak ada")
catat("Setelah tangani missing value", df)

Sisa missing value: tidak ada


## Langkah 3. Bersihkan baris

**Kenapa:**
- **Pasien meninggal atau hospice** (`discharge_disposition_id` = 11, 13, 14, 19, 20, 21 menurut `IDS_mapping.csv`) tidak mungkin dirawat ulang. Membiarkannya membuat label salah.
- **Gender "Unknown/Invalid"** hanya 3 baris, aman dibuang.
- **Pasien berulang**: satu pasien bisa muncul di banyak baris (kunjungan berbeda). Kalau baris pasien yang sama jatuh ke data latih dan data uji, model "mengintip" jawaban (kebocoran data). Solusi sederhana: satu kunjungan per pasien (kunjungan pertama).
  **Asumsi:** urutan `encounter_id` dianggap mewakili urutan waktu. Dokumentasi UCI tidak menjamin ini, jadi tulis di laporan.

In [4]:
KODE_MENINGGAL_HOSPICE = [11, 13, 14, 19, 20, 21]
n_mati = df["discharge_disposition_id"].isin(KODE_MENINGGAL_HOSPICE).sum()
df = df[~df["discharge_disposition_id"].isin(KODE_MENINGGAL_HOSPICE)]
print("Dibuang (meninggal/hospice):", n_mati)

n_gender = (df["gender"] == "Unknown/Invalid").sum()
df = df[df["gender"] != "Unknown/Invalid"]
print("Dibuang (gender tidak valid):", n_gender)

pasien_unik = df["patient_nbr"].nunique()
df = df.sort_values("encounter_id").drop_duplicates("patient_nbr", keep="first")
print("Pasien unik:", pasien_unik, "-> baris setelah satu kunjungan per pasien:", len(df))
catat("Setelah bersihkan baris", df)

Dibuang (meninggal/hospice): 2422
Dibuang (gender tidak valid): 3
Pasien unik: 69979 -> baris setelah satu kunjungan per pasien: 69979


## Langkah 4. Buang kolom yang tidak berguna

**Kenapa:**
- `encounter_id` dan `patient_nbr` hanya pengenal; model tidak boleh belajar dari nomor identitas.
- Kolom yang nilainya hanya satu (`examide`, `citoglipton`) tidak membawa informasi.
- Kolom obat yang hampir semuanya `No` (ambang **KEPUTUSAN 3**) hampir tidak membedakan pasien.

In [5]:
kolom_obat = list(df.loc[:, "metformin":"metformin-pioglitazone"].columns)
porsi_no = (df[kolom_obat] == "No").mean()
obat_buang = list(porsi_no[porsi_no >= AMBANG_OBAT_KONSTAN].index)
konstan = [c for c in df.columns if df[c].nunique() == 1]

print("Obat dibuang (>= {:.0%} 'No'): {}".format(AMBANG_OBAT_KONSTAN, obat_buang))
print("Kolom konstan:", konstan)

df = df.drop(columns=["encounter_id", "patient_nbr"] + obat_buang + [c for c in konstan if c not in obat_buang])
kolom_obat = [c for c in kolom_obat if c in df.columns]
print("Kolom obat yang dipertahankan:", kolom_obat)
catat("Setelah buang kolom", df)

Obat dibuang (>= 99% 'No'): ['nateglinide', 'chlorpropamide', 'acetohexamide', 'tolbutamide', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']
Kolom konstan: ['examide', 'citoglipton', 'glimepiride-pioglitazone']
Kolom obat yang dipertahankan: ['metformin', 'repaglinide', 'glimepiride', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'insulin']


## Langkah 5. Transformasi

**Kenapa:**
- `age` berupa rentang teks (`[70-80)`). Diubah ke angka titik tengah supaya urutannya terbaca model.
- `diag_1/2/3` punya ratusan kode ICD-9. Dikelompokkan ke kategori besar (sirkulasi, pernapasan, pencernaan, diabetes, cedera, dll.) mengikuti pengelompokan yang lazim dipakai pada data ini. Kode berawalan V/E menjadi "Lainnya".
- `admission_type_id`, `admission_source_id`, `discharge_disposition_id` adalah **kode**, bukan angka berurutan. Memperlakukannya sebagai angka menyesatkan, jadi dikelompokkan memakai arti di `IDS_mapping.csv`, lalu di-encoding sebagai kategori.
- Kolom obat bernilai `No/Steady/Up/Down` bukan ordinal alami (Up dan Down berlawanan arah), jadi nanti di-encoding sebagai kategori (one-hot), bukan angka urut.
- Label dibentuk dari `readmitted` sesuai **KEPUTUSAN 1**. Notebook ini memakai label **biner** (`<30` hari = 1; `>30` dan `NO` = 0), dengan alasan:
  1. Sumber datanya (Strack dkk., 2014) membagi hasil dengan cara yang sama: dirawat ulang dalam 30 hari atau "selain itu", yang mencakup readmisi setelah 30 hari dan tidak ada readmisi.
  2. Indeks LACE, dasar KBS di Tugas 1, memprediksi kematian atau readmisi tak terencana dalam 30 hari, jadi kasusnya memang berpusat pada batas 30 hari.
  3. Dua kelas membuat evaluasi di Tugas 3 lebih jelas: confusion matrix 2x2, serta recall dan presisi untuk satu kelas yang dicari.
  4. Label 3 kelas menambah kelas `>30` (31,8% dari 69.979 pasien) yang bukan fokus kasus ini, dan menuntut metrik rata-rata antar kelas serta perubahan KBS Tugas 1.

  Harga dari pilihan ini: kelas positif hanya 9,0% (6.283 pasien), jadi kelas tidak seimbang dan ditangani di langkah 8. Bila memakai 3 kelas, ubah `LABEL_MODE` di bagian Parameter dan jalankan ulang.

In [6]:
# --- usia: titik tengah rentang
peta_usia = {f"[{a}-{a+10})": a + 5 for a in range(0, 100, 10)}
df["age"] = df["age"].map(peta_usia)

# --- diagnosis ICD-9 ke kelompok
def kelompok_icd(kode):
    if kode == "Tidak ada":
        return "Tidak ada"
    if kode[0] in "VE":
        return "Lainnya"
    if kode.startswith("250"):
        return "Diabetes"
    x = float(kode)
    if 390 <= x <= 459 or x == 785: return "Sirkulasi"
    if 460 <= x <= 519 or x == 786: return "Pernapasan"
    if 520 <= x <= 579 or x == 787: return "Pencernaan"
    if 580 <= x <= 629 or x == 788: return "Genitourinari"
    if 140 <= x <= 239:             return "Neoplasma"
    if 710 <= x <= 739:             return "Muskuloskeletal"
    if 800 <= x <= 999:             return "Cedera"
    return "Lainnya"

for k in ["diag_1", "diag_2", "diag_3"]:
    df[k + "_grup"] = df[k].map(kelompok_icd)
df = df.drop(columns=["diag_1", "diag_2", "diag_3"])

# --- kode admisi dan pulang ke kelompok (arti dari IDS_mapping.csv)
def peta(kode, kamus, default="Lainnya"):
    for nama, daftar in kamus.items():
        if kode in daftar:
            return nama
    return default

TIPE_MASUK = {"Darurat": [1, 2, 7], "Elektif": [3], "Tidak diketahui": [5, 6, 8]}
SUMBER_MASUK = {"Rujukan": [1, 2, 3], "Transfer": [4, 5, 6, 10, 18, 19, 22, 25, 26],
                "UGD": [7], "Tidak diketahui": [9, 15, 17, 20, 21]}
TUJUAN_PULANG = {"Rumah": [1, 6, 8], "Pulang paksa": [7], "Tidak diketahui": [18, 25, 26],
                 "Fasilitas lain": [2, 3, 4, 5, 9, 10, 12, 15, 16, 17, 22, 23, 24, 27, 28, 29, 30]}

df["tipe_masuk"] = df["admission_type_id"].map(lambda k: peta(k, TIPE_MASUK))
df["sumber_masuk"] = df["admission_source_id"].map(lambda k: peta(k, SUMBER_MASUK))
df["tujuan_pulang"] = df["discharge_disposition_id"].map(lambda k: peta(k, TUJUAN_PULANG))
df = df.drop(columns=["admission_type_id", "admission_source_id", "discharge_disposition_id"])

# --- label
if LABEL_MODE == "biner":
    df["label"] = (df["readmitted"] == "<30").astype(int)
else:
    df["label"] = df["readmitted"].map({"NO": 0, ">30": 1, "<30": 2})
df = df.drop(columns=["readmitted"])

print("Distribusi label:", df["label"].value_counts(normalize=True).round(3).to_dict())
print(df[["age", "diag_1_grup", "tipe_masuk", "sumber_masuk", "tujuan_pulang", "label"]].head())
catat("Setelah transformasi", df)

Distribusi label: {0: 0.91, 1: 0.09}
    age diag_1_grup tipe_masuk sumber_masuk   tujuan_pulang  label
8    85   Sirkulasi    Darurat     Transfer           Rumah      0
9    95   Sirkulasi    Elektif     Transfer  Fasilitas lain      0
4    45   Neoplasma    Darurat          UGD           Rumah      0
10   45    Diabetes    Darurat          UGD           Rumah      0
5    55   Sirkulasi    Darurat      Rujukan           Rumah      0


## Langkah 6. Bagi data latih dan data uji

**Kenapa:** data uji harus "belum pernah dilihat" model. Karena langkah 3 sudah menjadikan satu baris = satu pasien,
pembagian acak aman dari kebocoran pasien. Pembagian `stratify` menjaga proporsi label sama di kedua bagian,
penting karena kelasnya tidak seimbang.

**Aturan penting:** semua langkah yang "belajar dari data" (batas outlier, scaling, encoding, penyeimbangan kelas) dilakukan **setelah** pembagian,
dan hanya dipelajari dari data latih. Kalau tidak, informasi data uji bocor ke latihan.

In [7]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["label"])
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print("Latih:", X_train.shape, " Uji:", X_test.shape)
print("Proporsi label latih:", y_train.value_counts(normalize=True).round(3).to_dict())
print("Proporsi label uji  :", y_test.value_counts(normalize=True).round(3).to_dict())

Latih: (55983, 31)  Uji: (13996, 31)
Proporsi label latih: {0: 0.91, 1: 0.09}
Proporsi label uji  : {0: 0.91, 1: 0.09}


## Langkah 7. Outlier, encoding, dan scaling (belajar dari data latih)

**Kenapa:**
- Kolom hitungan (`number_outpatient`, `number_emergency`, `number_inpatient`, `num_medications`) berekor panjang: sebagian besar kecil, sedikit sangat besar.
  Nilai besar itu bukan salah ukur, jadi tidak dibuang. Batasnya dipotong di persentil ke-99 **dari data latih**, lalu `log1p` untuk menjinakkan kemiringan.
- Kolom kategori (gender, ras, diagnosis, obat, hasil A1C, dll.) diubah ke one-hot karena model butuh angka.
- Kolom angka distandarisasi (rata-rata 0, simpangan baku 1) supaya skala besar (misalnya jumlah tes lab) tidak mendominasi.
- `A1Cresult` dan `max_glu_serum` bernilai `None` tetap sebagai kategori ("tes tidak dilakukan").

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

kolom_hitung = ["number_outpatient", "number_emergency", "number_inpatient", "num_medications"]

# --- sebelum: seberapa miring distribusinya
miring_sebelum = X_train[kolom_hitung].skew().round(2)

# --- batas dari DATA LATIH saja
batas = X_train[kolom_hitung].quantile(0.99)
def tangani_outlier(d):
    d = d.copy()
    d[kolom_hitung] = d[kolom_hitung].clip(upper=batas, axis=1)
    for k in ["number_outpatient", "number_emergency", "number_inpatient"]:
        d[k] = np.log1p(d[k])
    return d

Xtr = tangani_outlier(X_train)
Xte = tangani_outlier(X_test)

print("Kemiringan (skew) sebelum vs sesudah:")
print(pd.DataFrame({"sebelum": miring_sebelum, "sesudah": Xtr[kolom_hitung].skew().round(2)}))

kolom_kategori = Xtr.select_dtypes(include=["object", "string"]).columns.tolist()
kolom_angka = [c for c in Xtr.columns if c not in kolom_kategori]
print("\nKolom angka    :", kolom_angka)
print("Kolom kategori :", len(kolom_kategori), "kolom")

pra = ColumnTransformer([
    ("angka", StandardScaler(), kolom_angka),
    ("kategori", OneHotEncoder(handle_unknown="ignore", sparse_output=False), kolom_kategori),
])
Xtr_p = pd.DataFrame(pra.fit_transform(Xtr), columns=pra.get_feature_names_out(), index=Xtr.index)
Xte_p = pd.DataFrame(pra.transform(Xte), columns=pra.get_feature_names_out(), index=Xte.index)
print("\nUkuran setelah encoding dan scaling:", Xtr_p.shape, Xte_p.shape)

Kemiringan (skew) sebelum vs sesudah:
                   sebelum  sesudah
number_outpatient     9.67     2.77
number_emergency     23.20     3.62
number_inpatient      5.58     2.86
num_medications       1.45     1.10

Kolom angka    : ['age', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']
Kolom kategori : 22 kolom



Ukuran setelah encoding dan scaling: (55983, 191) (13996, 191)


## Langkah 8. Kelas tidak seimbang (hanya data latih)

**Kenapa:** kelas hasil readmisi `<30` hari jarang (sekitar 9% pada label biner). Model yang menebak "tidak" untuk semua pasien sudah akurat ±91%
tetapi tidak berguna. Karena itu **jangan memakai akurasi** sebagai metrik utama; pakai recall, F1, atau AUC.

Ada dua cara menangani (**KEPUTUSAN 4**), keduanya hanya pada data latih:
- *Oversampling acak* (diperagakan di bawah, dipakai di notebook ini): menggandakan contoh kelas langka.
- *class_weight='balanced'* saat melatih model (tidak mengubah data), bisa dipakai di Tugas 3.
Data uji dibiarkan apa adanya supaya evaluasinya jujur.

In [9]:
from sklearn.utils import resample

print("Distribusi kelas data latih sebelum:", y_train.value_counts().to_dict())

latih = Xtr_p.copy(); latih["label"] = y_train.values
terbanyak = latih["label"].value_counts().idxmax()
bagian = []
for kelas, grup in latih.groupby("label"):
    if kelas == terbanyak:
        bagian.append(grup)
    else:
        bagian.append(resample(grup, replace=True, n_samples=(latih["label"] == terbanyak).sum(), random_state=RANDOM_STATE))
latih_seimbang = pd.concat(bagian).sample(frac=1, random_state=RANDOM_STATE)

print("Distribusi kelas data latih setelah oversampling:", latih_seimbang["label"].value_counts().to_dict())
print("Distribusi kelas data uji (tidak diubah):", y_test.value_counts().to_dict())

Distribusi kelas data latih sebelum: {0: 50957, 1: 5026}


Distribusi kelas data latih setelah oversampling: {1: 50957, 0: 50957}
Distribusi kelas data uji (tidak diubah): {0: 12739, 1: 1257}


## Langkah 9. Analisis kepentingan fitur (tidak ada fitur yang dibuang)

**Yang dilakukan:**
- Random Forest dilatih pada data latih yang sudah diseimbangkan (langkah 8) hanya untuk mengukur kepentingan fitur; data uji tidak disentuh.
- Importance dari 191 kolom hasil one-hot digabung kembali ke 31 fitur mentah asalnya.
- Tidak ada fitur yang dibuang; dataset hasil langkah 7 dan 8 tidak berubah.

**Kenapa:**
- Reduksi data termasuk praproses; ini seleksi fitur jenis *embedded* (slide Minggu 7).
- Hasilnya dipakai untuk menentukan fitur yang layak jadi input pada sistem di Tugas 4.
- Fitur belum dibuang karena keputusan final menunggu perbandingan beberapa model di Tugas 3; ranking dari satu model belum cukup kuat.

**Hasil:**
- 15 fitur teratas menutup 79,5% total importance.
- Lima teratas: `tujuan_pulang` (17,46%), `number_inpatient` (6,44%), `medical_specialty` (6,14%), `num_lab_procedures` (5,37%), `diag_1_grup` (5,27%).
- `time_in_hospital`, `number_inpatient`, dan `number_diagnoses` termasuk kelompok teratas dan merupakan komponen indeks LACE (dasar Tugas 1).

**Keterbatasan:**
- Impurity importance condong ke fitur dengan banyak kategori (mis. `medical_specialty`, `payer_code`).
- Fitur yang berkorelasi (`diag_1_grup`, `diag_2_grup`, `diag_3_grup`) berbagi importance.
- Ini ranking dari satu model, bukan uji signifikansi dan bukan kesimpulan sebab-akibat.

**Dampak:** dataset tetap 191 kolom. Tugas 3 membandingkan ranking antar-model dan menguji 31 fitur melawan 15 teratas.

In [ ]:
# --- Langkah 9. Analisis kepentingan fitur (tidak ada fitur yang dibuang)
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

N_TOP_FITUR = 15   # jumlah fitur yang dilaporkan sebagai kelompok teratas

# data yang dipakai: HANYA data latih yang sudah diseimbangkan (langkah 8); data uji tidak disentuh
Xtr_bal = latih_seimbang.drop(columns=["label"])
ytr_bal = latih_seimbang["label"]

rf_imp = RandomForestClassifier(n_estimators=300, max_depth=12,
                                random_state=RANDOM_STATE, n_jobs=-1)
rf_imp.fit(Xtr_bal, ytr_bal)

# gabungkan importance 191 kolom hasil one-hot kembali ke 31 fitur mentah
encoder = pra.named_transformers_["kategori"]
nama_onehot = encoder.get_feature_names_out(kolom_kategori)
fitur_asal, idx = {}, 0
for kolom, kategori in zip(kolom_kategori, encoder.categories_):
    for nama in nama_onehot[idx:idx + len(kategori)]:
        fitur_asal["kategori__" + nama] = kolom
    idx += len(kategori)
for kolom in kolom_angka:
    fitur_asal["angka__" + kolom] = kolom
assert set(fitur_asal) == set(Xtr_bal.columns), "pemetaan kolom tidak lengkap"

imp = (pd.Series(rf_imp.feature_importances_, index=Xtr_bal.columns)
         .rename(index=fitur_asal).groupby(level=0).sum()
         .sort_values(ascending=False).to_frame("importance"))
imp["persen"] = (imp["importance"] / imp["importance"].sum() * 100).round(2)
imp["kumulatif_persen"] = imp["persen"].cumsum().round(2)

assert len(imp) == 31 and abs(imp["importance"].sum() - 1) < 1e-6
print(f"Fitur mentah: {len(imp)}. {N_TOP_FITUR} teratas menutup "
      f"{imp['persen'][:N_TOP_FITUR].sum():.1f}% total importance.\n")
print(imp[["persen", "kumulatif_persen"]].to_string())

fig, ax = plt.subplots(figsize=(8, 9))
warna = ["#2563eb" if i < N_TOP_FITUR else "#cbd5e1" for i in range(len(imp))]
ax.barh(imp.index[::-1], imp["importance"][::-1], color=warna[::-1])
ax.set_xlabel("Importance (Random Forest, data latih)")
ax.set_title(f"Ranking 31 fitur mentah (biru = {N_TOP_FITUR} teratas)")
plt.tight_layout()
plt.show()

# tidak ada fitur yang dibuang: dataset hasil langkah 7 dan 8 tidak berubah
print(f"\nData latih tetap {Xtr_p.shape[1]} kolom, data uji tetap {Xte_p.shape[1]} kolom (tidak ada fitur dibuang).")

## Langkah 10. Tabel sebelum dan sesudah, pemeriksaan akhir

In [10]:
tabel = pd.DataFrame(log)
tabel["baris_berkurang"] = tabel["baris"].shift(1) - tabel["baris"]
print(tabel.to_string(index=False))

print("\nHasil akhir:")
print(f"  Data latih : {Xtr_p.shape[0]} baris x {Xtr_p.shape[1]} fitur (setelah oversampling: {len(latih_seimbang)} baris)")
print(f"  Data uji   : {Xte_p.shape[0]} baris x {Xte_p.shape[1]} fitur")

# pemeriksaan otomatis
assert Xtr_p.isna().sum().sum() == 0 and Xte_p.isna().sum().sum() == 0, "masih ada missing value"
assert set(Xtr_p.columns) == set(Xte_p.columns), "kolom latih dan uji berbeda"
assert len(set(Xtr.index) & set(Xte.index)) == 0, "ada baris yang sama di latih dan uji"
print("\nPemeriksaan: tidak ada missing value, kolom latih = kolom uji, tidak ada baris tumpang tindih. OK")

                      langkah  baris  kolom  baris_berkurang
                  Data mentah 101766     50              NaN
Setelah tangani missing value 101745     49             21.0
      Setelah bersihkan baris  69979     49          31766.0
          Setelah buang kolom  69979     32              0.0
         Setelah transformasi  69979     32              0.0

Hasil akhir:
  Data latih : 55983 baris x 191 fitur (setelah oversampling: 101914 baris)
  Data uji   : 13996 baris x 191 fitur

Pemeriksaan: tidak ada missing value, kolom latih = kolom uji, tidak ada baris tumpang tindih. OK


## Langkah 11. Simpan hasil

In [11]:
os.makedirs("hasil_praproses", exist_ok=True)
Xtr_p.assign(label=y_train.values).to_csv("hasil_praproses/latih_praproses.csv", index=False)
latih_seimbang.to_csv("hasil_praproses/latih_seimbang.csv", index=False)
Xte_p.assign(label=y_test.values).to_csv("hasil_praproses/uji_praproses.csv", index=False)
print("Tersimpan di folder hasil_praproses/:", os.listdir("hasil_praproses"))

Tersimpan di folder hasil_praproses/: ['uji_praproses.csv', 'latih_praproses.csv', 'latih_seimbang.csv']


## Catatan

Tiap langkah di atas punya tiga bagian: **apa yang dilakukan**, **kenapa**, dan **dampaknya** (tabel langkah 10).
Asumsi yang dipakai:
1. `?` pada `diag_2` dan `diag_3` dianggap "tidak ada diagnosis tambahan".
2. Urutan `encounter_id` dianggap urutan waktu saat mengambil kunjungan pertama pasien.
3. Pengelompokan kode admisi dan pulang mengikuti arti di `IDS_mapping.csv`, dengan beberapa kode digabung ke "Lainnya".
4. Pengelompokan ICD-9 mengikuti pengelompokan lazim pada data ini; rentang kode perlu dicek ke sumber sebelum disitir.

Langkah berikutnya mengikuti slide tugas: pengolahan (klasifikasi) dan evaluasi (recall, F1, AUC, confusion matrix), lalu KBS dari model terbaik.